In [1]:
import os


In [2]:
os.chdir("../")
%pwd

'/Users/prasad/Learning/Projects/Kidney-Disease-Classification-Project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_including_top: bool
    params_weights: str
    params_classes: int
    params_epochs: int
    params_batch_size: int
    params_augmentation: bool


In [5]:
from cnnClassifierKidneyDisease.constants import *
from cnnClassifierKidneyDisease.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH) -> None:
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])



    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=config.root_dir,
            base_model_path=config.base_model_path,
            updated_base_model_path=config.updated_base_model_path,
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_including_top=self.params.INCLUDING_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES,
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_augmentation=self.params.AUGMENTATION
        )

        return prepare_base_model_config

In [11]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
from cnnClassifierKidneyDisease.logging import logger

In [9]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig) -> None:
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_including_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model) -> None:
        model.save(path)

    @staticmethod
    def _prepare_full_model(model: tf.keras.Model, config: PrepareBaseModelConfig, freeze_all: bool = True, freeze_till: int = None) -> tf.keras.Model:
        if (freeze_all):
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and freeze_till > 0:
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=config.params_classes, 
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=config.params_learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()

        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model, 
            config=self.config,
            freeze_all=True,
            freeze_till=None
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)




In [13]:
try:
    config_manager = ConfigurationManager()
    prepare_base_model_config = config_manager.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
except Exception as e:
    logger.error(f"An error occurred during base model preparation: {str(e)}")

[2026-05-17 11:31:48,193: INFO: common: YAML file loaded successfully: config/config.yaml]
[2026-05-17 11:31:48,196: INFO: common: YAML file loaded successfully: params.yaml]
[2026-05-17 11:31:48,196: INFO: common: Directory created successfully: artifacts]
[2026-05-17 11:31:48,197: INFO: common: Directory created successfully: artifacts/prepare_base_model]
58889256/58889256 [==============================] - 16s 0us/step
[2026-05-17 11:32:05,513: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]


/opt/homebrew/Caskroom/miniforge/base/envs/kidneyClassification/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [14]:
prepare_base_model.update_base_model()

[2026-05-17 11:33:17,356: WARNING: optimizer: At this time, the v2.11+ optimizer `tf.keras.optimizers.SGD` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.SGD`.]
[2026-05-17 11:33:17,358: WARNING: __init__: There is a known slowdown when using v2.11+ Keras optimizers on M1/M2 Macs. Falling back to the legacy Keras optimizer, i.e., `tf.keras.optimizers.legacy.SGD`.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling

/opt/homebrew/Caskroom/miniforge/base/envs/kidneyClassification/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
